# 01 - LLM Bootstrap Labelling

Uses an LLM (Gemini API) to **bootstrap** the HITL classifier seed labels.
Replaces — or precedes — the human seed step described in `classification_strategy.md` (Step 0).

**Inputs:** `llm_bootstrap_dataset.pkl` (~10 000 tweets carved out by `00_hitl_data_preparation.ipynb`).
This subset is disjoint from `base_dataset.pkl`, the HITL batches, and `inference_dataset.pkl` —
see `partition_ids.pkl` for the manifest.

**Pipeline:**
1. Load `llm_bootstrap_dataset.pkl`.
2. For each tweet, call the LLM with a prompt containing the per-category criteria.
3. Parse the JSON response, validate the label, retry on transient errors.
4. Checkpoint every N tweets to survive Colab disconnects.
5. Save the final CSV with the **same schema** as `hitl_review_batch_*.csv`
   so it drops straight into `02_hitl_training_loop.ipynb`.

Two cells are **placeholders** that must be filled in before running:
the category list + criteria, and the Gemini API key + model id.

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
%%time
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'google-genai'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
%%time
import json
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm
from google import genai
from google.genai import types

## Configuration

Tune the constants below per run. `SMOKE_TEST` keeps the pipeline cheap during development.

In [ ]:
%%time
# ── Run mode ────────────────────────────────────────────────────────────
SMOKE_TEST   = True       # True → label SMOKE_TEST_N tweets only; flip to False for the full run
SMOKE_TEST_N = 100

# Absolute ceiling on how many tweets are ever sent to the LLM in a single run.
# Enforced at three independent layers (dataframe truncation in Load Input, an
# assertion before the loop, and a per-call counter inside classify_tweet), so it
# binds even when SMOKE_TEST is False. The criteria in categories.md are still
# being tuned - raise this deliberately, never as a side effect of flipping
# SMOKE_TEST to False.
MAX_LLM_TWEETS = 1_000

# ── LLM ─────────────────────────────────────────────────────────────────
# Gemini model id. Cheapest first (per content/how-to/GEMINI_ERROR_HANDLING_SKILL.md):
#   'gemini-2.5-flash-lite'  — cheapest stable (DEFAULT); fine for one-shot classification
#   'gemini-2.5-flash'       — standard, more capable (thinking ON by default — see DISABLE_THINKING)
#   'gemini-2.5-pro'         — most capable; CANNOT disable thinking (min budget 128)
#   'gemini-2.0-flash-lite' / 'gemini-2.0-flash' — DEPRECATED (EOL ~June 2026)
MODEL_NAME  = 'gemini-2.5-flash-lite'
TEMPERATURE = 0.0         # deterministic classification; raise only if you want sampling diversity

# TOKENOPT_REF.md §4: always cap output tokens; 64-256 is the band for classification.
# 128 covers {label, confidence, one-sentence rationale}. If rationales come back
# truncated (they surface as PARSE_ERROR rows), raise this before anything else.
MAX_OUTPUT_TOKENS = 128

# Thinking is opt-out on gemini-2.5+ models (it inflates token cost a lot for tasks that
# don't need step-by-step reasoning). For one-shot classification we want it OFF; the
# config built below only attaches a ThinkingConfig when the chosen model is in the 2.5+
# family. Set False ONLY if you deliberately want the model to reason before answering.
DISABLE_THINKING = True

# ── Retry / backoff ─────────────────────────────────────────────────────
MAX_RETRIES     = 3
INITIAL_BACKOFF = 2.0     # seconds; doubled on each retry

# ── I/O ─────────────────────────────────────────────────────────────────
INPUT_PATH        = partitioned_folder / 'llm_bootstrap_dataset.pkl'
OUTPUT_CSV        = hitl_folder / 'llm_bootstrap_labels.csv'
OUTPUT_PKL        = hitl_folder / 'llm_bootstrap_labels_full.pkl'
CHECKPOINT_PREFIX = 'llm_bootstrap_checkpoint'
CHECKPOINT_EVERY  = 1_000  # save partial results every N tweets

## Categories and Criteria

The live taxonomy is **two labels**: `originality` (the tweet appeals to newness, creativity,
copying or theft **as a criterion for the value of art**) and `none` (the residual bucket —
no category in the current taxonomy applies).

- `CATEGORIES` is the closed list of allowed labels. The response schema in the API-key cell
  derives its `enum` from this list, so the model cannot return anything outside it.
- `CATEGORY_CRITERIA` is pasted **verbatim** from the *Criteria block* section of
  `categories.md`, which is the source of truth for the taxonomy — definitions, the decision
  test, worked examples, exclusions, and confidence calibration all live there.

**Sync obligation:** this notebook does not clone the repo on Colab (it only mounts Drive), so
`categories.md` is not readable at runtime and the criteria must live here as a literal. After
editing `categories.md`, re-paste its Criteria block into the cell below.

In [ ]:
%%time
# Closed label set. Pasted from the Label set table in categories.md.
# `none` is the residual bucket, not the negation of `originality` — it keeps its
# meaning when further categories are added.
CATEGORIES: list[str] = [
    'originality',
    'none',
]

# Pasted VERBATIM from the "Criteria block" section of categories.md.
# Edit categories.md first, then re-paste — not the other way round.
CATEGORY_CRITERIA: str = """
### originality

Definition. Originality in art refers to something which is non-trivially new in a work of
art. It relates to ideas of creativity (in the positive) and ideas about copying (in the
negative). Tweets that are referencing the importance of originality may mention creativity,
newness, difference to prior works, or theft, copying, tracing, plagiarising, replicating.
Label this category when that appeal is used as a criterion for the value of art.

Decision test. Label `originality` only when BOTH halves are present:
  (a) the tweet invokes newness or its absence — creative, original, novel, derivative,
      copy, steal, trace, plagiarise, replicate, rip off, regurgitate, unoriginal; AND
  (b) that invocation does evaluative work about art — it is offered as a reason the
      work, the practice, or the maker is good, bad, real, fake, valuable or worthless art.
Vocabulary alone is not enough. An evaluation of art on some other ground is not enough.

Positive examples.

1. -> originality, high confidence. Plagiarism is named outright as the condition under which
   the art would be unacceptable.
i don't mind ai art as long as it's not plagiarism...i also think some people a bit too lazy with it, seen some make a few posts with ai art that you can clearly see have flaws. yours is great, i really like it. but i feel like others should touch up the ai art before posting...

2. -> originality, medium confidence. The appeal is carried by the analogy rather than stated
   directly, and the scare-quoted "create" is doing the evaluative work.
and farmers learned from other farmers how to crow their crop and harvest. what's your point? the difference is that no farmer or artist is taking another's product and mixing it with yet another stolen product to "create" something. ai is just fancy photoshop for thieves.

3. -> originality, medium confidence. Theft and derivation are explicit, but the tweet is
   framed inside the jobs/automation argument, which competes for the tweet's main point.
sick and absolutely fucking tired of seeing people defending ai art "don't worry they're not taking away your jobs! people thought the same when cameras were invented!" homie that is not the point ai literally steals from artists, it takes whole ass aspects from existing pieces

Negative examples.

4. -> none. The objection is consent and payment, not that the output fails to be new.
they scraped every portfolio on the internet without asking and pay us nothing for it

5. -> none. Art is being evaluated, but on expression and emotion, not on newness.
ai art is empty. there is no human feeling behind any of it.

Exclusions — label `none`:
- Other value criteria. The tweet evaluates art on a ground other than newness: skill or
  effort, soul or emotion, meaning or understanding, beauty or formal qualities, morality,
  social or political function, or the experience of making it.
- Economic, consent or labour objections. The complaint is pay, permission, licensing or
  jobs rather than the work being derivative. If the tweet ALSO argues the output is not
  genuinely new, label originality instead.
- Novelty talk outside art. New models, products, research results, memes.
- Non-evaluative mention. Reporting, defining or quoting a copying dispute with no claim
  about artistic worth.

### none

No category above applies. This is the residual bucket — it is not a claim that the tweet
is unrelated to art or to AI.

Confidence.
0.8-1.0   Explicit: the vocabulary is present and the link to art's worth is stated outright.
0.5-0.79  Implicit: the appeal is inferred from framing, or shares the tweet with a competing
          theme (jobs, automation, consent) of equal or greater weight.
0.0-0.49  Contested: one plausible reading supports the label, another equally plausible
          reading does not.
Score confidence for whichever label you chose, including `none`.

In `rationale`, quote the phrase from the tweet that decided the label.
""".strip()

assert CATEGORIES, 'CATEGORIES is empty — fill it in before running.'
assert '<<< FILL IN' not in CATEGORY_CRITERIA, 'CATEGORY_CRITERIA still contains the placeholder.'
print(f'{len(CATEGORIES)} categories: {", ".join(CATEGORIES)}')
print(f'criteria: {len(CATEGORY_CRITERIA):,} chars (~{len(CATEGORY_CRITERIA)//4:,} tokens, re-sent per call)')

## API Key

**TODO — provide a Gemini API key before running.**

On Colab, store the key as a notebook secret named `GEMINI_API_KEY` (left sidebar → key icon)
and the cell below picks it up via `userdata.get`. Locally, set the `GEMINI_API_KEY`
environment variable. **Never hard-code the key in the notebook.**

In [ ]:
%%time
API_KEY = ''  # leave empty — populated below from Colab secrets / env var

if not RUNNING_LOCALLY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception as e:
        print(f'Colab userdata lookup failed: {e}')
else:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, 'GEMINI_API_KEY not set. Add it as a Colab secret or env var before running.'
assert MODEL_NAME, 'MODEL_NAME is empty — set it in the Configuration cell.'

client = genai.Client(api_key=API_KEY)

config_kwargs = dict(
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,   # TOKENOPT_REF.md §4
    response_mime_type='application/json',
)

# TOKENOPT_REF.md §5 — native structured output. The schema is enforced server-side, so
# the model cannot return a label outside CATEGORIES and cannot wrap its JSON in prose or
# markdown fences (the two things that produced PARSE_ERROR rows). The enum is derived
# from CATEGORIES, so adding a category needs no edit in this cell.
RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'label':      {'type': 'string', 'enum': CATEGORIES},
        'confidence': {'type': 'number'},
        'rationale':  {'type': 'string'},
    },
    'required': ['label', 'confidence', 'rationale'],
}
if 'response_json_schema' in types.GenerateContentConfig.model_fields:
    config_kwargs['response_json_schema'] = RESPONSE_SCHEMA
else:
    # older google-genai releases name the field response_schema
    config_kwargs['response_schema'] = RESPONSE_SCHEMA
if MODEL_NAME.startswith('gemini-2.5-pro'):
    # gemini-2.5-pro cannot turn thinking off; minimum thinking_budget is 128.
    config_kwargs['thinking_config'] = types.ThinkingConfig(thinking_budget=128)
    print(f'Note: {MODEL_NAME} cannot disable thinking; pinning thinking_budget=128')
elif DISABLE_THINKING and MODEL_NAME.startswith('gemini-2.5'):
    config_kwargs['thinking_config'] = types.ThinkingConfig(thinking_budget=0)
    print(f'Thinking disabled for {MODEL_NAME} (DISABLE_THINKING=True)')
elif not DISABLE_THINKING and MODEL_NAME.startswith('gemini-2.5'):
    print(f'WARNING: thinking is ENABLED for {MODEL_NAME} — expect higher token cost')
GEN_CONFIG = types.GenerateContentConfig(**config_kwargs)

print(f'LLM client ready: {MODEL_NAME}')

## Prompt and Response Schema

The LLM is asked to return a strict JSON object:
```
{"label": "<one of CATEGORIES>", "confidence": <float 0-1>, "rationale": "<one short sentence>"}
```
Anything else is treated as a parse error: it is retried, and on final failure the row is marked `PARSE_ERROR`.

In [ ]:
%%time
def build_prompt(tweet_text: str) -> str:
    return (
        'You are a tweet classifier for a research project on AI public trust.\n'
        'Classify the tweet into exactly one of the following categories:\n'
        f'{", ".join(CATEGORIES)}\n\n'
        'Per-category criteria:\n'
        f'{CATEGORY_CRITERIA}\n\n'
        'Return ONLY a JSON object with this exact schema (no prose, no markdown fences):\n'
        '{"label": "<one of the categories above>", '
        '"confidence": <number between 0 and 1>, '
        '"rationale": "<one short sentence>"}\n\n'
        f'Tweet:\n"""{tweet_text}"""'
    )

## Classification Function

Single-tweet wrapper: build prompt → call LLM → parse + validate JSON → retry on transient errors.

In [ ]:
%%time
PARSE_ERROR_RESULT = {'label': 'PARSE_ERROR', 'confidence': 0.0, 'rationale': ''}

# Innermost layer of the MAX_LLM_TWEETS cap. classify_tweet refuses to issue a request
# once the budget is spent, so the ceiling holds even if the dataframe is sliced
# elsewhere or this loop is re-entered by hand. The Run Classification cell resets the
# counter before it starts, which keeps Restart-and-Run-All clean.
_llm_calls_made = 0


def _spend_llm_budget() -> None:
    global _llm_calls_made
    if _llm_calls_made >= MAX_LLM_TWEETS:
        raise RuntimeError(
            f'MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,}) reached — refusing further LLM calls. '
            'Raise MAX_LLM_TWEETS in the Configuration cell if that is intended.'
        )
    _llm_calls_made += 1

def _strip_code_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
    return raw.strip()

def classify_tweet(text: str) -> dict:
    _spend_llm_budget()
    prompt = build_prompt(text)
    last_error = ''
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=GEN_CONFIG,
            )
            raw  = _strip_code_fences(resp.text or '')
            parsed = json.loads(raw)
            label = str(parsed.get('label', '')).strip()
            if label not in CATEGORIES:
                raise ValueError(f'label {label!r} not in CATEGORIES')
            return {
                'label': label,
                'confidence': float(parsed.get('confidence', 0.0)),
                'rationale': str(parsed.get('rationale', ''))[:500],
            }
        except Exception as e:
            last_error = f'{type(e).__name__}: {e}'
            if attempt + 1 < MAX_RETRIES:
                time.sleep(INITIAL_BACKOFF * (2 ** attempt))
    return {**PARSE_ERROR_RESULT, 'rationale': last_error[:500]}

## Load Input

In [ ]:
%%time
assert INPUT_PATH.exists(), f'Input not found: {INPUT_PATH}. Run 00_hitl_data_preparation.ipynb first.'
df = pd.read_pickle(INPUT_PATH)
print(f'Loaded {len(df):,} tweets from {INPUT_PATH.name}')

if SMOKE_TEST:
    df = df.sample(n=min(SMOKE_TEST_N, len(df)), random_state=42).reset_index(drop=True)
    print(f'SMOKE_TEST mode → using {len(df)} tweets')

# Hard cap — applied after the smoke-test slice so it binds with SMOKE_TEST either way.
if len(df) > MAX_LLM_TWEETS:
    df = df.sample(n=MAX_LLM_TWEETS, random_state=42).reset_index(drop=True)
    print(f'MAX_LLM_TWEETS cap → truncated to {len(df):,} tweets')
assert len(df) <= MAX_LLM_TWEETS, f'{len(df):,} rows exceeds MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,})'
print(f'Will send {len(df):,} tweets to {MODEL_NAME}')

for col in ('id', 'text', 'likes', 'retweets'):
    if col not in df.columns:
        df[col] = '' if col in ('id', 'text') else 0

df['text'] = df['text'].astype(str)

## Run Classification with Checkpointing

In [ ]:
%%time
assert len(df) <= MAX_LLM_TWEETS, f'{len(df):,} rows exceeds MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,})'
_llm_calls_made = 0   # reset the per-run budget so this cell is safely re-runnable

results: list[dict] = []
t0 = time.time()

for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc='LLM labelling'):
    classification = classify_tweet(row['text'])
    results.append({
        'id': row['id'],
        'text': row['text'],
        'likes': row.get('likes', 0),
        'retweets': row.get('retweets', 0),
        'predicted_label': classification['label'],
        'confidence': classification['confidence'],
        'rationale': classification['rationale'],
        'human_label': '',
    })
    if (len(results) % CHECKPOINT_EVERY) == 0:
        ckpt = hitl_folder / f'{CHECKPOINT_PREFIX}_{len(results)}.pkl'
        pd.DataFrame(results).to_pickle(ckpt)
        tqdm.tqdm.write(f'checkpoint → {ckpt.name} ({time.time()-t0:.0f}s elapsed)')

out_df = pd.DataFrame(results)
print(f'Done. Total time: {time.time()-t0:.1f}s')
print(out_df['predicted_label'].value_counts(dropna=False))

## Save Output

Two artifacts:
- **`llm_bootstrap_labels.csv`** — same schema as `hitl_review_batch_*.csv`, ready to drop into `02_hitl_training_loop.ipynb`.
- **`llm_bootstrap_labels_full.pkl`** — same data **plus** `confidence` and `rationale` columns for inspection.

In [ ]:
%%time
hitl_schema_cols = ['id', 'text', 'likes', 'retweets', 'predicted_label', 'human_label']
out_df[hitl_schema_cols].to_csv(OUTPUT_CSV, index=False)
out_df.to_pickle(OUTPUT_PKL)

n_errors = (out_df['predicted_label'] == 'PARSE_ERROR').sum()
print(f'Saved → {OUTPUT_CSV}')
print(f'Saved → {OUTPUT_PKL}')
print(f'PARSE_ERROR rows: {n_errors:,} / {len(out_df):,} ({n_errors/max(len(out_df),1):.1%})')

In [ ]:
# Disconnect from Colab runtime (no-op locally)
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    pass
